[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/mlflow-certified/notebooks/day-04-mlflow-ui.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · Mastering the MLflow UI — Comparing and Filtering Runs
**certified-journeys / mlflow-certified** · Practice session

> **Goal for today:** Log 6+ training runs with varied hyperparameters, filter them programmatically with `mlflow.search_runs()`, and export a comparison table to CSV for analysis.


In [ ]:
%pip install -q mlflow scikit-learn pandas matplotlib


## Step 1 · Set up MLflow tracking and import libraries

MLflow tracking is driven by a **tracking URI** — either a local directory (default: `./mlruns`) or a remote server.  
In Colab we use a local directory so everything stays in memory for the session.

| Concept | Meaning |
|---|---|
| `Experiment` | Logical grouping of related runs |
| `Run` | One training attempt with its params/metrics/artifacts |
| `mlruns/` | Default local directory that stores all run data |


In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# Use a local tracking directory so this works offline in Colab
mlflow.set_tracking_uri("./mlruns")

# Create (or get) an experiment — all Day 4 runs will live here
experiment_name = "day04-ui-comparison"
mlflow.set_experiment(experiment_name)

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {experiment_name}")


**What just happened?**

- `set_tracking_uri` tells MLflow where to write run data — a local `./mlruns` directory is created automatically.
- `set_experiment` creates the experiment if it doesn't exist; subsequent `mlflow.start_run()` calls attach to it.
- **The experiment ID is stable** — you can query it later with `mlflow.get_experiment_by_name()`.


## Step 2 · Prepare the dataset

We use **sklearn's breast cancer dataset** — 569 samples, 30 features, binary classification.  
We'll use this same split for every run so metrics are directly comparable.

**Rule of thumb:** Fix your random seed and train/test split *before* the experiment loop so hyperparameter differences — not data randomness — explain metric variance.


In [ ]:
# Load and split once — identical split for all 6 runs
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")
print(f"Classes: {data.target_names}")


**What just happened?**

- The breast cancer dataset is a classic binary classification benchmark — good for quick experiments.
- `random_state=42` ensures the split is reproducible across runs and machines.
- **Fixing the split means any metric difference across runs is purely due to hyperparameters**, not sampling variance.


## Step 3 · Log 6 training runs with varied hyperparameters

We log each run manually using `mlflow.start_run()` as a context manager.  
Inside each run we record:
- **Params** — `n_estimators`, `max_depth`, `learning_rate` (as a label for context)
- **Metrics** — `accuracy`, `f1_score`
- **Tags** — `model_type`, `dataset`

The Search API (used in Step 5) can filter on all of these.


In [ ]:
# 6 hyperparameter configurations to compare
configs = [
    {"n_estimators": 50,  "max_depth": 3,    "min_samples_split": 2},
    {"n_estimators": 100, "max_depth": 3,    "min_samples_split": 2},
    {"n_estimators": 100, "max_depth": 5,    "min_samples_split": 2},
    {"n_estimators": 200, "max_depth": 5,    "min_samples_split": 2},
    {"n_estimators": 100, "max_depth": None, "min_samples_split": 5},
    {"n_estimators": 200, "max_depth": None, "min_samples_split": 10},
]

run_ids = []  # collect run IDs for later queries

for i, cfg in enumerate(configs):
    with mlflow.start_run(run_name=f"rf-run-{i+1}") as run:
        # Train model
        model = RandomForestClassifier(
            n_estimators=cfg["n_estimators"],
            max_depth=cfg["max_depth"],
            min_samples_split=cfg["min_samples_split"],
            random_state=42,
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log params — must be strings or numbers; None becomes "None"
        mlflow.log_param("n_estimators", cfg["n_estimators"])
        mlflow.log_param("max_depth", str(cfg["max_depth"]))
        mlflow.log_param("min_samples_split", cfg["min_samples_split"])

        # Log metrics
        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)

        # Log a tag (tags are free-form strings, not typed)
        mlflow.set_tag("model_type", "RandomForest")
        mlflow.set_tag("dataset", "breast_cancer")

        run_ids.append(run.info.run_id)
        print(f"Run {i+1}: n_estimators={cfg['n_estimators']}, "
              f"max_depth={cfg['max_depth']}, "
              f"accuracy={acc:.4f}, f1={f1:.4f}")

print(f"\nLogged {len(run_ids)} runs.")


**What just happened?**

- Each `with mlflow.start_run()` block creates a new run in the experiment and auto-ends it on exit.
- **`log_param` vs `log_metric`**: params are logged once (hyperparameters); metrics can be logged multiple times (one per training step).
- Tags are flexible key-value strings — useful for filtering runs by `model_type`, `author`, or `dataset` without tying them to numeric comparisons.
- All run data is stored in `./mlruns/` as JSON/YAML files — no server needed.


## Step 4 · Reproduce a UI filter with `mlflow.search_runs()`

The MLflow UI lets you type filter expressions like:
```
metrics.accuracy > 0.85 AND params.n_estimators = '100'
```

`mlflow.search_runs()` accepts the **same syntax** and returns a Pandas DataFrame.

| Clause prefix | Example |
|---|---|
| `metrics.` | `metrics.accuracy > 0.90` |
| `params.` | `params.n_estimators = '100'` |
| `tags.` | `tags.model_type = 'RandomForest'` |
| `attributes.` | `attributes.status = 'FINISHED'` |

> Note: **param values are always strings** in filter expressions — use `= '100'` not `= 100`.


In [ ]:
# mlflow.search_runs returns a DataFrame — same syntax as the UI search bar
experiment = mlflow.get_experiment_by_name(experiment_name)

# Filter: accuracy > 0.85 AND n_estimators = '100' (mirrors the UI expression)
filter_expr = "metrics.accuracy > 0.85 AND params.n_estimators = '100'"

df_filtered = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=filter_expr,
    order_by=["metrics.accuracy DESC"],
)

# Show the columns we care about
display_cols = [
    "run_id",
    "params.n_estimators",
    "params.max_depth",
    "params.min_samples_split",
    "metrics.accuracy",
    "metrics.f1_score",
    "tags.mlflow.runName",
]

print(f"Filter: {filter_expr}")
print(f"Matching runs: {len(df_filtered)}\n")
print(df_filtered[[c for c in display_cols if c in df_filtered.columns]].to_string(index=False))


**What just happened?**

- `mlflow.search_runs()` returns a **Pandas DataFrame** — every param and metric becomes a column prefixed `params.` or `metrics.`.
- The filter syntax is **identical** to the MLflow UI search bar — no translation needed.
- **`order_by`** works like SQL ORDER BY; `DESC` puts the best run at the top.
- Param values are stored as strings internally, which is why the filter uses `= '100'` not `= 100`.


## Step 5 · Fetch all runs and build the full comparison table

Without a filter, `search_runs()` returns **all runs** in the experiment.  
We can then sort, filter, and plot entirely in Pandas — no UI required.

This is useful for:
- Automated reporting (CI pipeline comparisons)
- Plotting metric distributions across hundreds of runs
- Feeding results into a downstream decision script


In [ ]:
# Fetch all runs — no filter_string means "return everything"
df_all = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.accuracy DESC"],
)

# Build a clean comparison table with renamed columns
comparison = df_all[[
    "tags.mlflow.runName",
    "params.n_estimators",
    "params.max_depth",
    "params.min_samples_split",
    "metrics.accuracy",
    "metrics.f1_score",
]].copy()

comparison.columns = ["run_name", "n_estimators", "max_depth",
                      "min_samples_split", "accuracy", "f1_score"]
comparison["accuracy"] = comparison["accuracy"].round(4)
comparison["f1_score"]  = comparison["f1_score"].round(4)

print("All runs — sorted by accuracy:")
print(comparison.to_string(index=False))
print(f"\nBest run: {comparison.iloc[0]['run_name']} "
      f"(accuracy={comparison.iloc[0]['accuracy']})")


**What just happened?**

- `search_runs()` with no `filter_string` returns all finished runs — equivalent to clearing the search bar in the UI.
- We renamed columns for readability; the underlying DataFrame has MLflow's `params.*` / `metrics.*` prefix convention.
- **`order_by=["metrics.accuracy DESC"]`** puts the best model first — this mirrors the UI's column sort.
- The `comparison` DataFrame is now ready for export, plotting, or further pandas analysis.


## Step 6 · Export comparison table to CSV and visualize

Exporting to CSV gives you a snapshot of the experiment for:
- Sharing with teammates who don't have MLflow access
- Archiving experiment history in version control
- Further analysis in Excel, Google Sheets, or BI tools

After export, we'll plot accuracy by hyperparameter to see which setting drives the most improvement.


In [ ]:
# Export to CSV
csv_path = "day04_experiment_results.csv"
comparison.to_csv(csv_path, index=False)
print(f"Exported {len(comparison)} rows to {csv_path}")

# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: accuracy by n_estimators
comparison["n_estimators"] = comparison["n_estimators"].astype(int)
grouped = comparison.groupby("n_estimators")["accuracy"].max().reset_index()
axes[0].bar(grouped["n_estimators"].astype(str), grouped["accuracy"],
            color="steelblue", edgecolor="white")
axes[0].set_title("Max accuracy by n_estimators")
axes[0].set_xlabel("n_estimators")
axes[0].set_ylabel("accuracy")
axes[0].set_ylim(0.90, 1.0)

# Plot 2: accuracy vs f1 scatter
axes[1].scatter(comparison["accuracy"], comparison["f1_score"],
                s=80, color="coral", edgecolors="white", linewidths=0.8)
for _, row in comparison.iterrows():
    axes[1].annotate(row["run_name"], (row["accuracy"], row["f1_score"]),
                     fontsize=7, xytext=(3, 3), textcoords="offset points")
axes[1].set_title("Accuracy vs F1 Score")
axes[1].set_xlabel("accuracy")
axes[1].set_ylabel("f1_score")

plt.tight_layout()
plt.savefig("day04_comparison_plot.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved to day04_comparison_plot.png")


**What just happened?**

- `comparison.to_csv()` writes a standard CSV — you can re-read it with `pd.read_csv()` anytime.
- **The bar chart** shows how `n_estimators` affects peak accuracy — more trees often helps, but with diminishing returns.
- **The scatter plot** confirms that accuracy and F1 are highly correlated on this balanced dataset; divergence would signal a class-imbalance issue.
- You could log these plots back into MLflow with `mlflow.log_artifact("day04_comparison_plot.png")` to keep them with the experiment.


## Step 7 · Advanced search — using the MlflowClient API

For production workflows, the `MlflowClient` gives fine-grained control:  
list experiments, get run details, rename runs, and delete stale runs — all programmatically.

| Task | Client method |
|---|---|
| List experiments | `client.search_experiments()` |
| Get run details | `client.get_run(run_id)` |
| Set run tag | `client.set_tag(run_id, key, value)` |
| Delete a run | `client.delete_run(run_id)` |


In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# List all experiments in this tracking server
experiments = client.search_experiments()
print("Experiments on this server:")
for exp in experiments:
    print(f"  [{exp.experiment_id}] {exp.name}")

# Fetch the best run and inspect its details
best_run_id = comparison.iloc[0]["run_name"]  # run name, not ID — let's get the ID
best_run_id = df_all.sort_values("metrics.accuracy", ascending=False).iloc[0]["run_id"]

run_details = client.get_run(best_run_id)
print(f"\nBest run details:")
print(f"  Run ID: {run_details.info.run_id[:8]}...")
print(f"  Status: {run_details.info.status}")
print(f"  Params: {dict(run_details.data.params)}")
print(f"  Metrics: {dict(run_details.data.metrics)}")

# Tag the best run as champion — useful for filtering later
client.set_tag(best_run_id, "champion", "true")
print("\nTagged best run as champion=true")


**What just happened?**

- `MlflowClient` is the low-level API — it exposes operations not available through the top-level `mlflow.*` fluent API.
- `client.get_run(run_id)` returns full run metadata including params, metrics, tags, and artifact location.
- **Tagging the champion run** (`champion=true`) lets you find it later with `filter_string="tags.champion = 'true'"`.
- In production, CI pipelines use `MlflowClient` to automatically tag runs that beat a baseline threshold.


In [ ]:
# Challenge: Extend the search and analysis
#
# 1. Write a search_runs() call that finds runs where:
#    - f1_score > 0.93
#    - max_depth is NOT 'None' (i.e. params.max_depth != 'None')
#
# 2. Find the run with the highest f1_score and log an additional tag:
#    'best_f1' = 'true'
#    (use MlflowClient.set_tag)
#
# 3. Add a third metric 'precision' to a new run with n_estimators=150
#    and include it in your final comparison DataFrame.
#    Hint: from sklearn.metrics import precision_score

# Your solution here:
# df_challenge = mlflow.search_runs(
#     experiment_ids=[experiment.experiment_id],
#     filter_string="...",
# )


---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| `mlflow.start_run()` | Context manager — auto-ends the run on exit |
| `log_param` vs `log_metric` | Params are single values (hyperparams); metrics can be logged per step |
| Filter syntax | `metrics.X > N AND params.Y = 'val'` — same in UI and Python |
| `search_runs()` return type | Always a Pandas DataFrame — use `.sort_values()`, `.groupby()`, etc. |
| Param type in filters | Params are **strings** — use `= '100'` not `= 100` |
| `MlflowClient` | Low-level API for tagging, deleting, renaming runs |

> **Tip:** `mlflow.search_runs()` returns a Pandas DataFrame — you can sort, filter, and plot experiment results without ever opening the UI.

---
## What's next
**Day 5** → Autologging with sklearn, XGBoost, and PyTorch — let MLflow capture params/metrics/models automatically with a single `mlflow.autolog()` call.

Mark Day 4 complete in your [tracker](../index.html).
